# MNIST batch experiments: corrected weight wiring with full-dataset cosine

This notebook follows `MNIST batch.ipynb` and the repository training modules for **UNO**, **UNO-S**, **Gradient Surgery**, and retain-only **Retraining**.

The training path uses `vae_train.init` to create the model, optimizer, shuffled dataloaders, and fixed latent batch; the repository method processors perform every update; and the original logger, saver, and sample collector are used. The orthogonality and uniformity weights are passed in the corrected processor order. The only instrumented computation is a read-only full-dataset gradient cosine measurement. Its RNG and module-mode changes are restored before the original update runs.

As in the reference implementation, a request for 50 steps is rounded to one complete forget-loader epoch (53 updates for MNIST with batch size 128). The original log labels the first pre-update measurement as step 1. `State Step` below shifts that label to 0 so the untouched model is plotted at step 0.

In [156]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    repo_root = Path("/content/forget")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/pinakm9/forget.git", str(repo_root)],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pytorch-msssim"],
        check=True,
    )
    drive.mount("/content/drive")
    mnist_folder = Path("/content/MNIST")
    drive_mnist_folder = Path("/content/drive/MyDrive/Pinak/forget/MNIST")
    if not mnist_folder.exists():
        shutil.copytree(drive_mnist_folder, mnist_folder)
else:
    def find_repo_root(start: Path) -> Path:
        for candidate in (start, *start.parents):
            if (candidate / "modules" / "mnist").is_dir():
                return candidate
        raise FileNotFoundError("Could not find the repository root.")

    repo_root = find_repo_root(Path.cwd().resolve())
    mnist_folder = repo_root / "data" / "MNIST"

sys.path.insert(0, str(repo_root / "modules"))
sys.path.insert(0, str(repo_root / "modules" / "mnist"))

print(f"Repository: {repo_root}")
print(f"MNIST data: {mnist_folder}")

Repository: /Users/pinak/Documents/Github/forget
MNIST data: /Users/pinak/Documents/Github/forget/data/MNIST


In [157]:
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.autograd import grad
from tqdm.auto import tqdm

import classifier as cl
import datapipe
import utility as ut
import vae_loss as vl
import vae_ortho as vo
import vae_os as vos
import vae_ret as vr
import vae_surgery as vs
import vae_train as vt
import vae_viz as viz

device = ut.get_device()
print(f"Device: {device}")

Device: mps


## Parameters copied from `MNIST batch.ipynb`

No seed is added because the reference notebook does not set one. UNO and UNO-S use `orthogonality_weight=1e3`, `uniformity_weight=0`, and `forget_weight=0`, passed in the processor's expected order. Retraining and Surgery keep the 500 requested steps used in their reference cells; `vae_train.init` rounds those to 530 updates.

In [ ]:
num_experiments = 1
full_gradient_batch_size = 2048
experiment_folder = mnist_folder / "MNIST-Corrected-Training-Full-Cosine-Experiments"

params = {
    "model": str(mnist_folder / "vae" / "vae_200.pth"),
    "latent_dim": 2,
    "num_steps": 50,
    "batch_size": 128,
    "log_interval": 1,
    "collect_interval": "epoch",
    "save_steps": "epoch",
    "all_digits": list(range(10)),
    "forget_digit": 1,
    "classifier_path": str(mnist_folder / "classifiers" / "MNISTClassifier.pth"),
    "kl_weight": 1,
    "total_duration": None,
    "data_path": str(mnist_folder),
}

method_specs = {
    "Retraining": {
        "folder": "vae-ret",
        "num_steps": 50,
        "uniformity_weight": 0e3,
        "orthogonality_weight": 1e3,
        "forget_weight": 0.0,
    },
    "UNO": {
        "folder": "vae-o",
        "num_steps": 50,
        "uniformity_weight": 0e3,
        "orthogonality_weight": 1e3,
        "forget_weight": 0.0,
    },
    "UNO-S": {
        "folder": "vae-os",
        "num_steps": 50,
        "uniformity_weight": 0e3,
        "orthogonality_weight": 1e3,
        "forget_weight": 0.0,
    },
    "Surgery": {
        "folder": "vae-s",
        "num_steps": 50,
        "uniformity_weight": 0e3,
        "orthogonality_weight": 0.0,
        "forget_weight": None,
    },
}

assert Path(params["model"]).is_file(), params["model"]
assert Path(params["classifier_path"]).is_file(), params["classifier_path"]

# The reference notebook starts from an empty experiment directory.
if experiment_folder.exists():
    shutil.rmtree(experiment_folder)
experiment_folder.mkdir(parents=True)

print(json.dumps({"params": params, "methods": method_specs}, indent=2))

{
  "params": {
    "model": "/Users/pinak/Documents/Github/forget/data/MNIST/vae/vae_200.pth",
    "latent_dim": 2,
    "num_steps": 50,
    "batch_size": 128,
    "log_interval": 1,
    "collect_interval": "epoch",
    "save_steps": "epoch",
    "all_digits": [
      0,
      1,
      2,
      3,
      4,
      5,
      6,
      7,
      8,
      9
    ],
    "forget_digit": 1,
    "classifier_path": "/Users/pinak/Documents/Github/forget/data/MNIST/classifiers/MNISTClassifier.pth",
    "kl_weight": 1,
    "total_duration": null,
    "data_path": "/Users/pinak/Documents/Github/forget/data/MNIST"
  },
  "methods": {
    "Retraining": {
      "folder": "vae-ret",
      "num_steps": 50,
      "uniformity_weight": 0.0,
      "orthogonality_weight": 1000.0,
      "forget_weight": 0.0
    },
    "UNO": {
      "folder": "vae-o",
      "num_steps": 50,
      "uniformity_weight": 0.0,
      "orthogonality_weight": 1000.0,
      "forget_weight": 0.0
    },
    "UNO-S": {
      "folder": "vae-o

## Read-only full-dataset cosine diagnostic

Each retain and forget example contributes to its corresponding mean-loss gradient. The diagnostic uses evaluation mode to avoid BatchNorm-buffer updates, then restores every module mode and all Torch RNG states. `torch.autograd.grad` does not populate optimizer gradients.

In [159]:
def flatten_gradients(gradient_tensors):
    return torch.cat([tensor.reshape(-1) for tensor in gradient_tensors])


def capture_torch_rng_states(device):
    device = torch.device(device)
    states = {"cpu": torch.random.get_rng_state()}
    if torch.cuda.is_available():
        states["cuda"] = torch.cuda.get_rng_state_all()
    if device.type == "mps" and hasattr(torch.mps, "get_rng_state"):
        states["mps"] = torch.mps.get_rng_state()
    return states


def restore_torch_rng_states(states):
    torch.random.set_rng_state(states["cpu"])
    if "cuda" in states:
        torch.cuda.set_rng_state_all(states["cuda"])
    if "mps" in states:
        torch.mps.set_rng_state(states["mps"])


def full_dataset_gradient(net, loader, trainable_params, kl_weight):
    """Mean VAE-loss gradient over every sample, without changing training state."""
    rng_states = capture_torch_rng_states(net.device)
    module_modes = [(module, module.training) for module in net.modules()]
    total_samples = len(loader.dataset)
    flat_gradient = torch.zeros(
        sum(parameter.numel() for parameter in trainable_params),
        device=net.device,
    )

    try:
        net.eval()
        for images, _ in loader:
            images = images.view(images.shape[0], -1).to(net.device)
            reconstructed, mu, logvar = net(images)
            mean_loss_contribution = (
                vl.reconstruction_loss(reconstructed, images)
                + kl_weight * vl.kl_div(mu, logvar)
            ) / total_samples
            batch_gradient = grad(mean_loss_contribution, trainable_params)
            flat_gradient.add_(flatten_gradients(batch_gradient).detach())
    finally:
        for module, was_training in module_modes:
            module.training = was_training
        restore_torch_rng_states(rng_states)

    return flat_gradient


def measure_full_dataset_cosine(net, retain_loader, forget_loader, trainable_params, kl_weight):
    full_gr = full_dataset_gradient(net, retain_loader, trainable_params, kl_weight)
    full_gf = full_dataset_gradient(net, forget_loader, trainable_params, kl_weight)
    cosine = F.cosine_similarity(full_gr, full_gf, dim=0, eps=1e-12).clamp(-1.0, 1.0)
    return {
        "Full Dataset Gradient Dot Product": float(torch.dot(full_gr, full_gf)),
        "Full Dataset Retain Gradient Norm": float(torch.linalg.vector_norm(full_gr)),
        "Full Dataset Forget Gradient Norm": float(torch.linalg.vector_norm(full_gf)),
        "Full Dataset Cosine Similarity": float(cosine),
        "Absolute Full Dataset Cosine Similarity": float(cosine.abs()),
        "Full Dataset Cosine Distance": float(1.0 - cosine),
    }

## Corrected repository training loop with instrumentation

The ordering below matches the repository methods: obtain a shuffled retain/forget pair, run the original processor (which includes `optimizer.step()`), log its pre-update generated batch, save, and collect samples. The diagnostic is measured immediately before the original processor and is restored before training continues.

In [160]:
def train_instrumented_method(method, folder, full_gradient_batch_size, **cfg):
    if method not in method_specs:
        raise ValueError(f"Unknown method: {method}")

    train_mode = "orthogonal-surgery" if method == "UNO-S" else "orthogonal"
    init_orthogonality_weight = cfg["orthogonality_weight"]

    (
        net,
        dataloader,
        optim,
        z_random,
        identifier,
        sample_dir,
        checkpoint_dir,
        epoch_length,
        epochs,
        actual_num_steps,
        save_steps,
        collect_interval,
        log_interval,
        csv_file,
        method_device,
        grid_size,
    ) = vt.init(
        model=cfg["model"],
        folder=str(folder),
        num_steps=cfg["num_steps"],
        batch_size=cfg["batch_size"],
        latent_dim=cfg["latent_dim"],
        save_steps=cfg["save_steps"],
        collect_interval=cfg["collect_interval"],
        log_interval=cfg["log_interval"],
        kl_weight=cfg["kl_weight"],
        uniformity_weight=cfg["uniformity_weight"],
        orthogonality_weight=init_orthogonality_weight,
        forget_weight=cfg["forget_weight"],
        all_digits=cfg["all_digits"],
        forget_digit=cfg["forget_digit"],
        classifier_path=cfg["classifier_path"],
        train_mode=train_mode,
        data_path=cfg["data_path"],
    )

    trainable_params = ut.get_trainable_params(net)
    processor_weights = (
        cfg["kl_weight"],
        cfg["orthogonality_weight"],
        cfg["uniformity_weight"],
        cfg["forget_weight"] if cfg["forget_weight"] is not None else 0.0,
    )
    surgery_weights = (cfg["kl_weight"], cfg["uniformity_weight"])

    if method == "UNO":
        process_odd = vo.get_processor(
            net, trainable_params, identifier, z_random, processor_weights,
            optim, cfg["all_digits"], cfg["forget_digit"],
        )
        process_even = process_odd
        log_results = vo.get_logger(identifier, csv_file, log_interval)
    elif method == "UNO-S":
        process_odd = vo.get_processor(
            net, trainable_params, identifier, z_random, processor_weights,
            optim, cfg["all_digits"], cfg["forget_digit"],
        )
        process_even = vs.get_processor(
            net, trainable_params, identifier, z_random, surgery_weights,
            optim, cfg["all_digits"], cfg["forget_digit"],
        )
        log_results = vo.get_logger(identifier, csv_file, log_interval)
    elif method == "Surgery":
        process_odd = vs.get_processor(
            net, trainable_params, identifier, z_random, surgery_weights,
            optim, cfg["all_digits"], cfg["forget_digit"],
        )
        process_even = process_odd
        log_results = vo.get_logger(identifier, csv_file, log_interval)
    else:
        process_odd = vr.get_processor(
            net, trainable_params, identifier, z_random, processor_weights,
            optim, cfg["all_digits"], cfg["forget_digit"],
        )
        process_even = process_odd
        log_results = vr.get_logger(identifier, csv_file, log_interval)

    save = vt.get_saver(net, save_steps, checkpoint_dir, epoch_length)
    collect_samples = vt.get_collector(sample_dir, collect_interval, grid_size)

    full_retain_loader = torch.utils.data.DataLoader(
        dataloader["retain"].dataset,
        batch_size=full_gradient_batch_size,
        shuffle=False,
    )
    full_forget_loader = torch.utils.data.DataLoader(
        dataloader["forget"].dataset,
        batch_size=full_gradient_batch_size,
        shuffle=False,
    )

    diagnostic_rows = []
    global_step = 0
    for _ in tqdm(range(1, epochs + 1), desc=f"{method} epochs"):
        paired_batches = zip(dataloader["retain"], dataloader["forget"])
        for (img_retain, _), (img_forget, _) in paired_batches:
            diagnostic = measure_full_dataset_cosine(
                net,
                full_retain_loader,
                full_forget_loader,
                trainable_params,
                cfg["kl_weight"],
            )

            global_step += 1
            process_batch = (
                process_even
                if method == "UNO-S" and global_step % 2 == 0
                else process_odd
            )
            (
                rec_loss,
                kl_loss,
                uniformity_loss,
                orthogonality_loss,
                generated_img,
                logits,
                elapsed_time,
            ) = process_batch(img_retain, img_forget)

            if method == "Surgery":
                total_loss = (
                    rec_loss
                    + cfg["kl_weight"] * kl_loss
                    + cfg["uniformity_weight"] * uniformity_loss
                )
            else:
                total_loss = (
                    rec_loss
                    + cfg["kl_weight"] * kl_loss
                    + cfg["uniformity_weight"] * uniformity_loss
                    + cfg["orthogonality_weight"] * orthogonality_loss
                )

            # This seemingly unused batch is part of the original loop and is retained
            # because creating its shuffled iterator consumes RNG.
            real_img, _ = next(iter(dataloader["original"]))
            real_img = real_img.view(real_img.shape[0], -1).to(method_device)
            log_results(
                step=global_step,
                losses=[
                    rec_loss,
                    kl_loss,
                    uniformity_loss,
                    orthogonality_loss,
                    total_loss,
                ],
                elapsed_time=elapsed_time,
                real_img=real_img,
                generated_img=generated_img,
                logits=logits,
            )
            save(step=global_step)
            collect_samples(generated_img, step=global_step)

            class_fractions = (
                torch.bincount(logits.detach().argmax(dim=1), minlength=10).float()
                / logits.shape[0]
            )
            diagnostic_rows.append(
                {
                    "State Step": global_step - 1,
                    "Training Log Step": global_step,
                    "Method": method,
                    "1 Fraction": float(class_fractions[cfg["forget_digit"]]),
                    **diagnostic,
                }
            )

    diagnostic_frame = pd.DataFrame(diagnostic_rows)
    diagnostic_path = Path(checkpoint_dir) / "full_dataset_cosine.csv"
    diagnostic_frame.to_csv(diagnostic_path, index=False)

    reference_log = pd.read_csv(csv_file)
    reference_log["State Step"] = reference_log["Step"] - 1
    combined_log = reference_log.merge(
        diagnostic_frame.drop(columns=["Training Log Step", "1 Fraction"]),
        on="State Step",
        how="left",
    )
    combined_log.to_csv(Path(checkpoint_dir) / "training_log_with_full_cosine.csv", index=False)

    viz.summarize_training(
        folder=str(folder),
        total_duration=cfg["total_duration"],
    )
    return combined_log, {
        "requested_steps": cfg["num_steps"],
        "actual_steps": actual_num_steps,
        "epoch_length": epoch_length,
        "epochs": epochs,
        "fixed_latent_shape": tuple(z_random.shape),
    }

## Run the four reference methods

This cell can be slow because the requested diagnostic traverses the entire retain and forget datasets before every update. Training minibatches remain shuffled exactly as in the reference.

In [ ]:
logs = {}
run_metadata = {}

for method, method_spec in method_specs.items():
    method_params = params | {
        key: value for key, value in method_spec.items() if key != "folder"
    }
    for experiment_id in range(num_experiments):
        run_folder = (
            experiment_folder
            / method_spec["folder"]
            / f"expr-{experiment_id}"
        )
        logs[(method, experiment_id)], run_metadata[(method, experiment_id)] = (
            train_instrumented_method(
                method=method,
                folder=run_folder,
                full_gradient_batch_size=full_gradient_batch_size,
                **method_params,
            )
        )

print(json.dumps({str(key): value for key, value in run_metadata.items()}, indent=2))
print(f"Results written to: {experiment_folder}")

Retraining epochs:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/pinak/Documents/Github/forget/modules/mnist/vae_viz.py:261: UserWarning: Data has no positive values, and therefore cannot be log-scaled.
  plt.savefig(f"{folder}/evolution.png", bbox_inches="tight")


Time taken by summarize_training is 0.1771 seconds


UNO epochs:   0%|          | 0/1 [00:00<?, ?it/s]

Time taken by summarize_training is 0.2040 seconds


UNO-S epochs:   0%|          | 0/1 [00:00<?, ?it/s]

Time taken by summarize_training is 0.2199 seconds


Surgery epochs:   0%|          | 0/10 [00:00<?, ?it/s]

## Plot full-dataset $|\cos(g_r,g_f)|$ and semilog `1 Fraction`

Exact zeros cannot be represented on a logarithmic axis, so they are displayed at half of one sample (`0.5 / batch_size`) and marked with ×. The CSV retains the exact zero values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
zero_display_floor = 0.5 / params["batch_size"]

for (method, experiment_id), frame in logs.items():
    label = method if num_experiments == 1 else f"{method} expr-{experiment_id}"
    steps = frame["State Step"]

    axes[0].plot(
        steps,
        frame["Absolute Full Dataset Cosine Similarity"],
        label=label,
    )

    one_fraction = frame["1 Fraction"]
    plotted_fraction = one_fraction.mask(one_fraction == 0, zero_display_floor)
    line = axes[1].semilogy(steps, plotted_fraction, label=label)[0]
    zero_mask = one_fraction == 0
    if zero_mask.any():
        axes[1].scatter(
            steps[zero_mask],
            np.full(int(zero_mask.sum()), zero_display_floor),
            marker="x",
            color=line.get_color(),
            s=25,
        )

axes[0].set_title("Full-dataset absolute cosine similarity")
axes[0].set_xlabel("Completed updates (State Step)")
axes[0].set_ylabel(r"$|\cos(g_r,g_f)|$")
axes[0].set_ylim(-0.02, 1.02)
axes[0].grid(alpha=0.3)

axes[1].axhline(0.02, color="black", linestyle="--", linewidth=1.2, label="0.02 threshold")
axes[1].set_title("Generated digit-1 fraction")
axes[1].set_xlabel("Completed updates (State Step)")
axes[1].set_ylabel("1 Fraction (log scale)")
axes[1].grid(alpha=0.3, which="both")

for axis in axes:
    axis.legend()

fig.tight_layout()
plot_path = experiment_folder / "full_dataset_absolute_cosine_and_one_fraction.png"
fig.savefig(plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved plot: {plot_path}")

## Generated images after training

The original collector decodes one fixed latent batch per run and saves it at every completed epoch. The following cell displays the final saved grid for each method without generating a different latent batch.

In [ ]:
from IPython.display import Image, display

for method, method_spec in method_specs.items():
    sample_folder = experiment_folder / method_spec["folder"] / "expr-0" / "samples"
    sample_paths = sorted(
        sample_folder.glob("sample_*.jpg"),
        key=lambda path: int(path.stem.split("_")[-1]),
    )
    if not sample_paths:
        print(f"{method}: no saved samples found")
        continue
    print(f"{method}: {sample_paths[-1]}")
    display(Image(filename=str(sample_paths[-1])))

### Reading the replicated log

- `training_log.csv` is the untouched reference-format log: its step 1 measurement is produced before update 1, although the update occurs before the row is written.
- `full_dataset_cosine.csv` contains the added read-only diagnostic with zero-based `State Step`.
- `training_log_with_full_cosine.csv` combines both and is used by the plots.
- The last reference row describes the state before the last update. The epoch checkpoint contains the model after that last update. This off-by-one behavior is retained for consistency with the original logger.
- The archived `MNIST-Experiments/vae-o` curves were produced with the historical swapped-weight bug and should not be treated as results from the corrected UNO objective. With the bug fixed, `orthogonality_weight=1000` controls the cosine-squared term and `uniformity_weight=0` disables the generated-class penalty.